# Business Capability Mapping Tester

This notebook tests capability matching for **selected organizations** (e.g., DGs 23 and 24).

**Features:**
- Load pre-built org and capability graphs (no reprocessing)
- Select specific organizations by code prefix (e.g., "23", "24")
- Run bipartite matching with LLM judge
- Store results and embeddings in graph
- Generate Excel report with match details and justifications

**Outputs:**
- `./output/capability_mapping_report.xlsx` - Detailed Excel report
- `./output/selected_org_matches.json` - JSON match results
- FalkorDB graph: `org_capability_matches`

In [1]:
# =============================================================================
# CONFIGURATION
# =============================================================================

import os
import sys
sys.path.insert(0, '.')

# ═══════════════════════════════════════════════════════════════════════════
# SELECT ORGANIZATIONS TO MAP
# ═══════════════════════════════════════════════════════════════════════════

# Specify org code prefixes to include (e.g., ["23", "24"] for DGs 23 and 24)
# Set to None or [] to map ALL organizations
#SELECTED_ORG_PREFIXES = ["23", "24"]
SELECTED_ORG_PREFIXES = ["10"]

# ═══════════════════════════════════════════════════════════════════════════
# GRAPH LOADING
# ═══════════════════════════════════════════════════════════════════════════

LOAD_FROM = "local"  # "local" or "falkordb"

# Local paths
ORG_GRAPH_PATH = "./graph_data/organization_graph.json"
CAP_GRAPH_PATH = "./graph_data/capability_graph.json"

# ═══════════════════════════════════════════════════════════════════════════
# FALKORDB SETTINGS
# ═══════════════════════════════════════════════════════════════════════════

USE_REAL_FALKORDB = True
FALKORDB_URL_BASE = "redis://@r-6jissuruar.instance-zeqb0ww84.hc-v8noonp0c.europe-west1.gcp.f2e0a955bb84.cloud:52346"
ORG_GRAPH_NAME = "org_hierarchy"
CAP_GRAPH_NAME = "capability_map"
MATCH_GRAPH_NAME = "org_capability_matches"

# ═══════════════════════════════════════════════════════════════════════════
# MATCHING SETTINGS
# ═══════════════════════════════════════════════════════════════════════════

USE_LLM_JUDGE = True           # Use LLM for evaluation with justification
USE_MOCK_LLM = False           # False = real Claude API
#LLM_MODEL = "claude-sonnet-4-20250514"
LLM_MODEL = "claude-haiku-4-5-20251001"

MIN_SEMANTIC_SCORE = 0.25      # Minimum semantic similarity
MIN_LLM_SCORE = 0.5            # Minimum LLM judge score
ENABLE_HIERARCHICAL_FALLBACK = True  # Try higher cap levels if no leaf match
TOP_K_CANDIDATES = 5           # Candidates per org

USE_MOCK_EMBEDDER = False      # False = real HuggingFace embeddings

# ═══════════════════════════════════════════════════════════════════════════
# OUTPUT
# ═══════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = "./output"
EXCEL_REPORT_PATH = f"{OUTPUT_DIR}/capability_mapping_report.xlsx"
JSON_RESULTS_PATH = f"{OUTPUT_DIR}/selected_org_matches.json"

print("Configuration:")
print(f"  Selected orgs: {SELECTED_ORG_PREFIXES or 'ALL'}")
print(f"  Load from: {LOAD_FROM}")
print(f"  LLM Judge: {USE_LLM_JUDGE}")
print(f"  Mock LLM: {USE_MOCK_LLM}")
print(f"  Min semantic: {MIN_SEMANTIC_SCORE}")
print(f"  Min LLM: {MIN_LLM_SCORE}")
print(f"  Hierarchical fallback: {ENABLE_HIERARCHICAL_FALLBACK}")

Configuration:
  Selected orgs: ['10']
  Load from: local
  LLM Judge: True
  Mock LLM: False
  Min semantic: 0.25
  Min LLM: 0.5
  Hierarchical fallback: True


In [2]:
# =============================================================================
# SETUP: API KEY
# =============================================================================

os.environ["ANTHROPIC_API_KEY"] = ""

print("✓ API key configured")

✓ API key configured


## Step 1: Load Pre-built Graphs

Loads organizational and capability graphs without reprocessing.

In [3]:
# =============================================================================
# LOAD GRAPHS
# =============================================================================

import json
import networkx as nx
from bipartite_matcher import BipartiteCapabilityMatcher, MatcherConfig, MockEmbedder

org_graph = None
cap_graph = None

if LOAD_FROM == "local":
    print("Loading graphs from local JSON files...")
    
    # Load org graph
    if os.path.exists(ORG_GRAPH_PATH):
        with open(ORG_GRAPH_PATH, 'r', encoding='utf-8') as f:
            org_data = json.load(f)
        
        org_graph = nx.DiGraph()
        for node in org_data.get("nodes", []):
            node_id = node.pop("id")
            org_graph.add_node(node_id, **node)
        for edge in org_data.get("edges", []):
            org_graph.add_edge(edge["source"], edge["target"])
        
        print(f"  ✓ Org graph: {org_graph.number_of_nodes()} nodes, {org_graph.number_of_edges()} edges")
    else:
        print(f"  ⚠ Org graph not found: {ORG_GRAPH_PATH}")
    
    # Load cap graph
    if os.path.exists(CAP_GRAPH_PATH):
        with open(CAP_GRAPH_PATH, 'r', encoding='utf-8') as f:
            cap_data = json.load(f)
        
        cap_graph = nx.DiGraph()
        for node in cap_data.get("nodes", []):
            node_id = node.pop("id")
            cap_graph.add_node(node_id, **node)
        for edge in cap_data.get("edges", []):
            cap_graph.add_edge(edge["source"], edge["target"])
        
        print(f"  ✓ Cap graph: {cap_graph.number_of_nodes()} nodes, {cap_graph.number_of_edges()} edges")
    else:
        print(f"  ⚠ Cap graph not found: {CAP_GRAPH_PATH}")

elif LOAD_FROM == "falkordb":
    print("Loading graphs from FalkorDB...")
    
    from falkordb import FalkorDB
    client = FalkorDB.from_url(FALKORDB_URL_BASE)
    
    # Load org graph
    org_db = client.select_graph(ORG_GRAPH_NAME)
    org_graph = nx.DiGraph()
    
    result = org_db.query("MATCH (n) RETURN n")
    for record in result.result_set:
        node = record[0]
        props = dict(node.properties)
        node_id = props.pop("node_id", str(node.id))
        org_graph.add_node(node_id, **props)
    
    result = org_db.query("MATCH (a)-[r]->(b) RETURN a.node_id, b.node_id")
    for record in result.result_set:
        if record[0] and record[1]:
            org_graph.add_edge(record[0], record[1])
    
    print(f"  ✓ Org graph: {org_graph.number_of_nodes()} nodes")
    
    # Load cap graph
    cap_db = client.select_graph(CAP_GRAPH_NAME)
    cap_graph = nx.DiGraph()
    
    result = cap_db.query("MATCH (n) RETURN n")
    for record in result.result_set:
        node = record[0]
        props = dict(node.properties)
        node_id = props.pop("node_id", str(node.id))
        cap_graph.add_node(node_id, **props)
    
    result = cap_db.query("MATCH (a)-[r]->(b) RETURN a.node_id, b.node_id")
    for record in result.result_set:
        if record[0] and record[1]:
            cap_graph.add_edge(record[0], record[1])
    
    print(f"  ✓ Cap graph: {cap_graph.number_of_nodes()} nodes")

if org_graph and cap_graph:
    print("\n✓ Both graphs loaded successfully")
else:
    print("\n⚠ Failed to load graphs")

Loading graphs from local JSON files...
  ✓ Org graph: 529 nodes, 495 edges
  ✓ Cap graph: 383 nodes, 374 edges

✓ Both graphs loaded successfully


## Step 2: Filter Selected Organizations

In [4]:
# =============================================================================
# FILTER SELECTED ORGANIZATIONS
# =============================================================================

def get_org_leaves(graph):
    """Get leaf nodes (no children)."""
    return [n for n in graph.nodes() if graph.out_degree(n) == 0]

def filter_orgs_by_prefix(graph, prefixes):
    """Filter org nodes by code prefix."""
    if not prefixes:
        return list(graph.nodes())
    
    selected = []
    for node_id in graph.nodes():
        for prefix in prefixes:
            if node_id.startswith(prefix):
                selected.append(node_id)
                break
    return selected

# Get all org leaves
all_leaves = get_org_leaves(org_graph)
print(f"Total org leaves: {len(all_leaves)}")

# Filter by selected prefixes
if SELECTED_ORG_PREFIXES:
    # Get all nodes matching prefixes (including non-leaves for context)
    selected_nodes = filter_orgs_by_prefix(org_graph, SELECTED_ORG_PREFIXES)
    
    # Get leaves from selected
    selected_leaves = [n for n in selected_nodes if n in all_leaves]
    
    print(f"\nSelected prefixes: {SELECTED_ORG_PREFIXES}")
    print(f"Selected nodes: {len(selected_nodes)}")
    print(f"Selected leaves (for matching): {len(selected_leaves)}")
    
    # Show selected orgs
    print("\nSelected organizations:")
    for node_id in sorted(selected_leaves)[:10]:
        data = org_graph.nodes[node_id]
        print(f"  [{node_id}] {data.get('name', 'N/A')[:50]}")
    if len(selected_leaves) > 10:
        print(f"  ... and {len(selected_leaves) - 10} more")
else:
    selected_leaves = all_leaves
    print("\nNo filter applied - will match ALL org leaves")

target_orgs = selected_leaves

Total org leaves: 454

Selected prefixes: ['10']
Selected nodes: 71
Selected leaves (for matching): 65

Selected organizations:
  [10-10] 10-10
  [10-20] 10-20
  [10-2010] 10-2010
  [10A10] 10A10
  [10A1010] 10A1010
  [10A1020] 10A1020
  [10A20] 10A20
  [10A2010] 10A2010
  [10A30] 10A30
  [10A3010] 10A3010
  ... and 55 more


## Step 3: Setup Matcher & Run Matching

In [5]:
# =============================================================================
# SETUP LLM
# =============================================================================

from llm import create_llm

LLM_MODEL = "claude-haiku-4-5-20251001"
llm = create_llm(use_mock=USE_MOCK_LLM, model=LLM_MODEL)

C:\Users\marci\anaconda3\envs\DATAENLIGHT_AI_ARCH_PATTERNS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ LLM: claude-haiku-4-5-20251001


In [6]:
# =============================================================================
# CREATE CUSTOM MATCHER FOR SELECTED ORGS
# =============================================================================

from bipartite_matcher import (
    BipartiteCapabilityMatcher, 
    MatcherConfig, 
    MockEmbedder, 
    HuggingFaceEmbedder,
    MatchResult
)

class SelectiveCapabilityMatcher(BipartiteCapabilityMatcher):
    """Extended matcher that operates on selected org nodes only."""
    
    def run_selective_matching(self, target_org_ids: list) -> list:
        """Run matching for specific org IDs only."""
        if self.config.verbose:
            print("\n" + "═" * 70)
            print("SELECTIVE CAPABILITY MATCHING")
            print("═" * 70)
        
        if not self._org_embeddings:
            self.compute_embeddings()
        
        cap_leaves = self._get_leaves(self.cap_graph)
        
        if self.config.verbose:
            print(f"\nTarget orgs: {len(target_org_ids)}")
            print(f"Cap leaves: {len(cap_leaves)}")
            print("\n--- Matching ---")
        
        self.results = []
        self.unmatched_orgs = []
        
        for i, org_id in enumerate(target_org_ids):
            if org_id not in self.org_graph.nodes():
                print(f"  ⚠ {org_id} not in graph")
                continue
            
            result = self.match_org_to_capabilities(org_id, cap_leaves)
            self.results.append(result)
            
            if result.unmatched:
                self.unmatched_orgs.append(org_id)
            
            if self.config.verbose:
                org_name = self.org_graph.nodes[org_id].get("name", org_id)[:35]
                if result.matches:
                    best = result.best_match()
                    print(f"[{i+1}/{len(target_org_ids)}] {org_id}: {org_name}")
                    print(f"  → {best['capability_name'][:35]} ({best['combined_score']:.3f} {best['match_type']})")
                else:
                    print(f"[{i+1}/{len(target_org_ids)}] {org_id}: {org_name} [UNMATCHED]")
        
        # Hierarchical fallback
        if self.config.enable_hierarchical_fallback and self.unmatched_orgs:
            if self.config.verbose:
                print(f"\n--- Hierarchical Fallback ({len(self.unmatched_orgs)} unmatched) ---")
            
            for level in range(1, self.config.max_fallback_levels + 1):
                if not self.unmatched_orgs:
                    break
                
                max_cap_level = max(self.cap_graph.nodes[n].get("level", 0) for n in self.cap_graph.nodes())
                target_level = max_cap_level - level
                if target_level < 0:
                    break
                
                higher_caps = self._get_nodes_at_level(self.cap_graph, target_level)
                
                if self.config.verbose:
                    print(f"\nTrying level {target_level} ({len(higher_caps)} capabilities)...")
                
                still_unmatched = []
                for org_id in self.unmatched_orgs:
                    result = self.match_org_to_capabilities(org_id, higher_caps, fallback_level=level)
                    
                    for r in self.results:
                        if r.org_id == org_id:
                            r.matches = result.matches
                            r.unmatched = result.unmatched
                            r.fallback_level = level
                            break
                    
                    if result.unmatched:
                        still_unmatched.append(org_id)
                    elif self.config.verbose:
                        best = result.best_match()
                        org_name = self.org_graph.nodes[org_id].get("name", org_id)[:30]
                        print(f"  {org_id}: → {best['capability_name'][:30]} (L{target_level})")
                
                self.unmatched_orgs = still_unmatched
        
        self._build_bipartite_graph()
        
        matched = sum(1 for r in self.results if not r.unmatched)
        if self.config.verbose:
            print(f"\n✓ Matching complete: {matched}/{len(self.results)} matched")
        
        return self.results

# Create matcher
embedder = MockEmbedder() if USE_MOCK_EMBEDDER else HuggingFaceEmbedder()

config = MatcherConfig(
    min_semantic_score=MIN_SEMANTIC_SCORE,
    min_llm_score=MIN_LLM_SCORE,
    top_k_candidates=TOP_K_CANDIDATES,
    use_llm_judge=USE_LLM_JUDGE,
    enable_hierarchical_fallback=ENABLE_HIERARCHICAL_FALLBACK,
    sync_to_falkordb=USE_REAL_FALKORDB,
    bipartite_graph_name=MATCH_GRAPH_NAME,
    verbose=True
)

matcher = SelectiveCapabilityMatcher(org_graph, cap_graph, llm, embedder, config)
print("✓ Matcher created")

C:\Users\marci\anaconda3\envs\DATAENLIGHT_AI_ARCH_PATTERNS\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\marci\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4348.74it/s]
M

✓ Embedder: all-mpnet-base-v2
✓ Matcher created


In [7]:
# =============================================================================
# RUN MATCHING
# =============================================================================

match_results = matcher.run_selective_matching(target_orgs)

# Print summary
matcher.print_summary()


══════════════════════════════════════════════════════════════════════
SELECTIVE CAPABILITY MATCHING
══════════════════════════════════════════════════════════════════════

Computing embeddings...
  Org embeddings: 454
  Cap embeddings: 383

Target orgs: 65
Cap leaves: 328

--- Matching ---
[1/65] 10-10: 10-10
  → Governance Structures (0.558 MODERATE)
[2/65] 10-20: 10-20
  → Governance Structures (0.654 STRONG)
[3/65] 10-2010: 10-2010 [UNMATCHED]
[4/65] 10A10: 10A10 [UNMATCHED]
[5/65] 10A1010: 10A1010 [UNMATCHED]
[6/65] 10A1020: 10A1020 [UNMATCHED]
[7/65] 10A20: 10A20 [UNMATCHED]
[8/65] 10A2010: 10A2010 [UNMATCHED]
[9/65] 10A30: 10A30
  → Service Desk Operation (0.613 MODERATE)
[10/65] 10A3010: 10A3010 [UNMATCHED]
[11/65] 10A3020: 10A3020 [UNMATCHED]
[12/65] 10A40: 10A40 [UNMATCHED]
[13/65] 10A50: 10A50 [UNMATCHED]
[14/65] 10A5010: 10A5010 [UNMATCHED]
[15/65] 10A5020: 10A5020
  → IT Service Life Cycle Management (0.624 MODERATE)


KeyboardInterrupt: 

## Step 4: Store Results & Vectorize in Graph

In [10]:
# =============================================================================
# STORE RESULTS IN GRAPHS
# =============================================================================

def store_matches_in_graphs(matcher, results, org_graph, cap_graph):
    """Store match results and embeddings in both NetworkX and FalkorDB."""
    
    print("\nStoring matches in graphs...")
    
    # Update org nodes with their matches
    for result in results:
        if result.org_id in org_graph.nodes():
            best = result.best_match()
            if best:
                org_graph.nodes[result.org_id]["matched_capability"] = best["capability_id"]
                org_graph.nodes[result.org_id]["match_score"] = best["combined_score"]
                org_graph.nodes[result.org_id]["match_type"] = best["match_type"]
                org_graph.nodes[result.org_id]["match_justification"] = best["justification"]
    
    print(f"  ✓ Updated {len(results)} org nodes in NetworkX")
    
    # Sync to FalkorDB
    if USE_REAL_FALKORDB:
        try:
            from falkordb import FalkorDB
            client = FalkorDB.from_url(FALKORDB_URL_BASE)
            
            # Sync bipartite matches
            count = matcher.sync_to_falkordb(client)
            
            # Update org nodes with match info
            org_db = client.select_graph(ORG_GRAPH_NAME)
            for result in results:
                best = result.best_match()
                if best:
                    query = """
                    MATCH (n {node_id: $node_id})
                    SET n.matched_capability = $cap_id,
                        n.match_score = $score,
                        n.match_type = $match_type,
                        n.match_justification = $justification
                    """
                    org_db.query(query, {
                        "node_id": result.org_id,
                        "cap_id": best["capability_id"],
                        "score": best["combined_score"],
                        "match_type": best["match_type"],
                        "justification": best["justification"]
                    })
            
            print(f"  ✓ Synced to FalkorDB")
        except Exception as e:
            print(f"  ⚠ FalkorDB sync failed: {e}")
    
    return len(results)

store_matches_in_graphs(matcher, match_results, org_graph, cap_graph)


Storing matches in graphs...
  ✓ Updated 15 org nodes in NetworkX

Syncing matches to FalkorDB...
✓ Synced 13 matches to FalkorDB
  ✓ Synced to FalkorDB


15

In [11]:
# =============================================================================
# VECTORIZE MATCH EMBEDDINGS
# =============================================================================

def vectorize_matches(matcher, org_graph):
    """Store embeddings for matched orgs."""
    
    print("\nVectorizing org embeddings...")
    
    count = 0
    for org_id, embedding in matcher._org_embeddings.items():
        if org_id in org_graph.nodes():
            org_graph.nodes[org_id]["embedding"] = embedding
            count += 1
    
    print(f"  ✓ Stored {count} embeddings in NetworkX")
    
    # Sync to FalkorDB
    if USE_REAL_FALKORDB:
        try:
            from falkordb import FalkorDB
            client = FalkorDB.from_url(FALKORDB_URL_BASE)
            org_db = client.select_graph(ORG_GRAPH_NAME)
            
            synced = 0
            for org_id, embedding in matcher._org_embeddings.items():
                try:
                    query = """
                    MATCH (n {node_id: $node_id})
                    SET n.embedding = $embedding
                    """
                    org_db.query(query, {"node_id": org_id, "embedding": embedding})
                    synced += 1
                except:
                    pass
            
            print(f"  ✓ Synced {synced} embeddings to FalkorDB")
        except Exception as e:
            print(f"  ⚠ FalkorDB sync failed: {e}")

vectorize_matches(matcher, org_graph)


Vectorizing org embeddings...
  ✓ Stored 454 embeddings in NetworkX
  ✓ Synced 454 embeddings to FalkorDB


## Step 5: Generate Excel Report

In [12]:
# =============================================================================
# GENERATE EXCEL REPORT
# =============================================================================

import pandas as pd
from pathlib import Path

def get_capability_path(cap_graph, cap_id):
    """Get full hierarchy path for a capability."""
    path = [cap_graph.nodes[cap_id].get("name", cap_id)]
    current = cap_id
    while True:
        preds = list(cap_graph.predecessors(current))
        if not preds:
            break
        parent = preds[0]
        parent_name = cap_graph.nodes[parent].get("name", parent)
        if "ROOT" not in parent_name.upper():
            path.insert(0, parent_name)
        current = parent
    return " > ".join(path)

def generate_excel_report(results, org_graph, cap_graph, output_path):
    """Generate detailed Excel report with multiple sheets."""
    
    print("\nGenerating Excel report...")
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Sheet 1: Match Summary
    summary_rows = []
    for result in results:
        org_data = org_graph.nodes.get(result.org_id, {})
        best = result.best_match()
        
        row = {
            "Org Code": result.org_id,
            "Org Name": result.org_name,
            "Org Level": result.org_level,
            "Status": "MATCHED" if not result.unmatched else "UNMATCHED",
            "Fallback Level": result.fallback_level,
            "# Matches": len(result.matches),
        }
        
        if best:
            cap_data = cap_graph.nodes.get(best["capability_id"], {})
            row.update({
                "Best Cap ID": best["capability_id"],
                "Best Cap Name": best["capability_name"],
                "Cap Level": best["capability_level"],
                "Cap Type": cap_data.get("node_type", ""),
                "Cap Path": get_capability_path(cap_graph, best["capability_id"]),
                "Semantic Score": best["semantic_score"],
                "LLM Score": best["llm_score"],
                "Combined Score": best["combined_score"],
                "Match Type": best["match_type"],
                "Justification": best["justification"],
                "Key Overlaps": best.get("key_overlaps", ""),
                "Gaps": best.get("gaps", "")
            })
        
        summary_rows.append(row)
    
    df_summary = pd.DataFrame(summary_rows)
    
    # Sheet 2: All Matches (detailed)
    detail_rows = []
    for result in results:
        org_data = org_graph.nodes.get(result.org_id, {})
        
        for match in result.matches:
            cap_data = cap_graph.nodes.get(match["capability_id"], {})
            
            row = {
                "Org Code": result.org_id,
                "Org Name": result.org_name,
                "Org Activities": "; ".join(org_data.get("activities", [])[:3]),
                "Org Refined Desc": org_data.get("refined_description", "")[:200],
                "Cap ID": match["capability_id"],
                "Cap Name": match["capability_name"],
                "Cap Level": match["capability_level"],
                "Cap Type": cap_data.get("node_type", ""),
                "Cap Path": get_capability_path(cap_graph, match["capability_id"]),
                "Cap Description": cap_data.get("refined_description", cap_data.get("description", ""))[:200],
                "Cap Keywords": cap_data.get("capability_keywords", ""),
                "Semantic Score": match["semantic_score"],
                "LLM Score": match["llm_score"],
                "Combined Score": match["combined_score"],
                "Match Type": match["match_type"],
                "Justification": match["justification"],
                "Key Overlaps": match.get("key_overlaps", ""),
                "Gaps": match.get("gaps", "")
            }
            detail_rows.append(row)
    
    df_details = pd.DataFrame(detail_rows)
    
    # Sheet 3: Unmatched Orgs
    unmatched_rows = []
    for result in results:
        if result.unmatched:
            org_data = org_graph.nodes.get(result.org_id, {})
            row = {
                "Org Code": result.org_id,
                "Org Name": result.org_name,
                "Org Level": result.org_level,
                "Activities": "; ".join(org_data.get("activities", [])),
                "Refined Description": org_data.get("refined_description", "")
            }
            unmatched_rows.append(row)
    
    df_unmatched = pd.DataFrame(unmatched_rows)
    
    # Sheet 4: Statistics
    stats = {
        "Metric": [
            "Total Orgs Processed",
            "Matched",
            "Unmatched",
            "Match Rate",
            "Avg Semantic Score",
            "Avg LLM Score",
            "Avg Combined Score",
            "Strong Matches",
            "Moderate Matches",
            "Weak Matches",
            "Fallback Level 0 (Leaf)",
            "Fallback Level 1",
            "Fallback Level 2+"
        ],
        "Value": []
    }
    
    matched = [r for r in results if not r.unmatched]
    all_scores = [r.best_match()["combined_score"] for r in matched if r.best_match()]
    sem_scores = [r.best_match()["semantic_score"] for r in matched if r.best_match()]
    llm_scores = [r.best_match()["llm_score"] for r in matched if r.best_match()]
    
    type_counts = {"STRONG": 0, "MODERATE": 0, "WEAK": 0}
    fallback_counts = {0: 0, 1: 0, 2: 0}
    for r in matched:
        best = r.best_match()
        if best:
            t = best.get("match_type", "UNKNOWN")
            type_counts[t] = type_counts.get(t, 0) + 1
        fb = r.fallback_level
        if fb >= 2:
            fallback_counts[2] += 1
        else:
            fallback_counts[fb] = fallback_counts.get(fb, 0) + 1
    
    stats["Value"] = [
        len(results),
        len(matched),
        len(results) - len(matched),
        f"{len(matched)/len(results)*100:.1f}%" if results else "0%",
        f"{sum(sem_scores)/len(sem_scores):.3f}" if sem_scores else "N/A",
        f"{sum(llm_scores)/len(llm_scores):.3f}" if llm_scores else "N/A",
        f"{sum(all_scores)/len(all_scores):.3f}" if all_scores else "N/A",
        type_counts.get("STRONG", 0),
        type_counts.get("MODERATE", 0),
        type_counts.get("WEAK", 0),
        fallback_counts.get(0, 0),
        fallback_counts.get(1, 0),
        fallback_counts.get(2, 0)
    ]
    
    df_stats = pd.DataFrame(stats)
    
    # Write to Excel
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df_summary.to_excel(writer, sheet_name='Match Summary', index=False)
        df_details.to_excel(writer, sheet_name='All Matches Detail', index=False)
        df_unmatched.to_excel(writer, sheet_name='Unmatched Orgs', index=False)
        df_stats.to_excel(writer, sheet_name='Statistics', index=False)
    
    print(f"✓ Excel report saved: {output_path}")
    print(f"  Sheets: Match Summary, All Matches Detail, Unmatched Orgs, Statistics")
    
    return output_path

generate_excel_report(match_results, org_graph, cap_graph, EXCEL_REPORT_PATH)


Generating Excel report...
✓ Excel report saved: ./output/capability_mapping_report.xlsx
  Sheets: Match Summary, All Matches Detail, Unmatched Orgs, Statistics


'./output/capability_mapping_report.xlsx'

## Step 6: Save JSON Results

In [ ]:
# =============================================================================
# SAVE JSON RESULTS
# =============================================================================

def save_json_results(results, output_path):
    """Save match results to JSON."""
    
    output = {
        "metadata": {
            "selected_prefixes": SELECTED_ORG_PREFIXES,
            "total_orgs": len(results),
            "matched": sum(1 for r in results if not r.unmatched),
            "unmatched": sum(1 for r in results if r.unmatched),
            "config": {
                "use_llm_judge": USE_LLM_JUDGE,
                "min_semantic_score": MIN_SEMANTIC_SCORE,
                "min_llm_score": MIN_LLM_SCORE,
                "hierarchical_fallback": ENABLE_HIERARCHICAL_FALLBACK
            }
        },
        "matches": [r.to_dict() for r in results]
    }
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    
    print(f"\n✓ JSON results saved: {output_path}")

save_json_results(match_results, JSON_RESULTS_PATH)

# Also save bipartite graph
matcher.save_bipartite_graph("./graph_data/selected_bipartite_matches.json")

## Summary

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "═" * 70)
print("CAPABILITY MAPPING SUMMARY")
print("═" * 70)

matched = [r for r in match_results if not r.unmatched]
unmatched = [r for r in match_results if r.unmatched]

print(f"\nSelected orgs: {SELECTED_ORG_PREFIXES or 'ALL'}")
print(f"Total processed: {len(match_results)}")
print(f"Matched: {len(matched)} ({len(matched)/len(match_results)*100:.1f}%)" if match_results else "N/A")
print(f"Unmatched: {len(unmatched)}")

if matched:
    type_counts = {}
    for r in matched:
        best = r.best_match()
        if best:
            t = best["match_type"]
            type_counts[t] = type_counts.get(t, 0) + 1
    
    print("\nMatch types:")
    for t, c in sorted(type_counts.items()):
        print(f"  {t}: {c}")

print(f"\nOutputs:")
print(f"  Excel: {EXCEL_REPORT_PATH}")
print(f"  JSON: {JSON_RESULTS_PATH}")
print(f"  FalkorDB: {MATCH_GRAPH_NAME if USE_REAL_FALKORDB else 'N/A'}")

print("\n" + "═" * 70)